In [2]:
import pandas as pd
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def scrape_phong_vu(so_luong_can_lay):
    print(f"Bắt đầu quét dữ liệu -> Mục tiêu: {so_luong_can_lay} sản phẩm.")
    
    chrome_options = Options()
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-notifications")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    results = []
    page = 1

    try:
        while len(results) < so_luong_can_lay:
            url_listing = f"https://phongvu.vn/c/laptop?page={page}"
            print(f"--- Đang quét trang {page} ---")
            driver.get(url_listing)
            
            try:
                WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.product-card")))
            except:
                print("Không tìm thấy sản phẩm hoặc đã hết trang.")
                break

            for _ in range(4):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(1)

            cards = driver.find_elements(By.CSS_SELECTOR, "div.product-card")
            
            for card in cards:
                if len(results) >= so_luong_can_lay:
                    break
                
                try:
                    # 1. Lấy thông tin cơ bản
                    ten_raw = card.find_element(By.CSS_SELECTOR, "div.att-product-card-title h3").text.strip()
                    ten = ten_raw.split('(')[0].strip()
                    url_sp = card.find_element(By.XPATH, ".//a").get_attribute("href")
                    
                    try:
                        gia_ht_raw = card.find_element(By.CSS_SELECTOR, "div.att-product-detail-latest-price").text.strip()
                        gia_hien_tai = re.sub(r"[^\d]", "", gia_ht_raw)
                    except:
                        gia_hien_tai = "0"

                    try:
                        gia_goc_raw = card.find_element(By.XPATH, ".//del | .//*[contains(@class, 'retail-price')]").text.strip()
                        gia_goc = re.sub(r"[^\d]", "", gia_goc_raw)
                    except:
                        gia_goc = gia_hien_tai 

                    # LẤY TOÀN BỘ TEXT CỦA CARD ĐỂ BÓC TÁCH THÔNG SỐ (Kỹ thuật Regex)
                    card_text = card.text

                    # 2. Bóc tách RAM (Tìm các số 8, 16, 24, 32, 64 đi kèm chữ GB)
                    ram_match = re.search(r'\b(4|8|16|24|32|64)\s*GB\b', card_text, re.IGNORECASE)
                    ram = ram_match.group(0).upper() if ram_match else "N/A"

                    # 3. Bóc tách SSD/Ổ cứng (Tìm 256GB, 512GB hoặc 1TB, 2TB)
                    ssd_match = re.search(r'\b(128|256|512)\s*GB\b|\b(1|2)\s*TB\b', card_text, re.IGNORECASE)
                    ssd = ssd_match.group(0).upper() if ssd_match else "N/A"

                    # 4. Bóc tách CPU (Bắt các định dạng như i7-13620H, U7-255U, R5 340, Core 5...)
                    cpu_pattern = r'\b(i[3579]-?\w+|U[579]-?\w+|R[3579]\s?\w+|Core\s?[357]\s?\w+|Apple\s?M\d\w*)\b'
                    cpu_match = re.search(cpu_pattern, card_text, re.IGNORECASE)
                    cpu = cpu_match.group(0) if cpu_match else "N/A"

                    # 5. Bóc tách Card đồ họa GPU (Bắt RTX, GTX, Intel Graphics, Radeon...)
                    gpu_pattern = r'\b(RTX\s?\d{4}|GTX\s?\d{4}|Radeon\w*|Intel\s?Graphics|Arc\s?Graphics|Apple\s?GPU)\b'
                    gpu_match = re.search(gpu_pattern, card_text, re.IGNORECASE)
                    gpu = gpu_match.group(0) if gpu_match else "Onboard"

                    # Chỉ lưu nếu có giá hợp lệ
                    if ten and url_sp and gia_hien_tai != "0":
                        results.append({
                            "Tên sản phẩm": ten,
                            "CPU": cpu,
                            "RAM": ram,
                            "SSD": ssd,
                            "VGA/GPU": gpu,
                            "Giá gốc": gia_goc,
                            "Giá hiện tại": gia_hien_tai,
                            "URL": url_sp
                        })
                        print(f"[{len(results)}] {ten} | CPU: {cpu} | RAM: {ram} | GPU: {gpu}")
                        
                except Exception as e:
                    continue
            
            page += 1
            time.sleep(1)

    finally:
        driver.quit()

    return pd.DataFrame(results)

if __name__ == "__main__":
    SO_LUONG = 50
    OUTPUT_FILE = "Laptop_PhongVu.csv"

    df = scrape_phong_vu(SO_LUONG)

    if not df.empty:
        df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
        print(f"\nTHÀNH CÔNG! Đã lưu {len(df)} sản phẩm.")

Bắt đầu quét dữ liệu -> Mục tiêu: 50 sản phẩm.
--- Đang quét trang 1 ---
[1] Laptop Msi Katana B14WEK-286VN | CPU: i5-14450HX | RAM: 16GB | GPU: Onboard
[2] Laptop Asus Vivobook S14 S3407CA-LY068WS | CPU: N/A | RAM: 16GB | GPU: Onboard
[3] Laptop Lenovo IdeaPad Slim 3 15ARP10 - 83K700GDVN | CPU: R5 7535HS | RAM: 16GB | GPU: Onboard
[4] Laptop HP 15-fc0024AU - D0BH2PA | CPU: N/A | RAM: 16GB | GPU: Onboard
[5] Laptop HP Victus 15 fa2731TX | CPU: i5-13420H | RAM: 16GB | GPU: Onboard
[6] Laptop Lenovo V15 G5 IRL - 83HF00BXVN | CPU: i7-13620H | RAM: 16GB | GPU: Intel Graphics
[7] Laptop HP OmniBook 7 14-fr0027TU - C1MN1PA | CPU: U7-255U | RAM: 16GB | GPU: Intel Graphics
[8] Laptop Lenovo Legion 5 15IPH11 - 83RW0023VN | CPU: U7-356H | RAM: 16GB | GPU: RTX 5060
[9] Laptop HP OmniBook 5 16 ag1069AU - BZ7T1PA | CPU: R5 340 | RAM: 16GB | GPU: Onboard
[10] Laptop HP OmniBook 7 14-fs0042TU - C1MP3PA | CPU: Core 5 210H | RAM: 24GB | GPU: Intel Graphics
[11] Laptop HP 15-fd1289TU - C2CV8PA | CPU: U7